# Буткемп, день 13 — событийный движок: жизненный цикл ордера, очередь и бюджет смещений

Формула дня звучит как предупреждение: *«повтор, который детерминирован, всё ещё может быть неверным;
работа движка — сделать неверное невозможным»*. Дни 11–12 строили филлы из **интенсивности**:
вероятность срабатывания котировки за шаг бралась как $\lambda(\delta)=Ae^{-k\delta}$, а сам филл
разыгрывался как независимая монетка. Это описание удобно, но у него есть дефект: такая модель **не различает, кто стоял впереди нас в очереди** и почему филл вообще
случился. Мы переходим от интенсивностей к **событиям ордеров**: движок крутится на очереди событий,
каждый ордер проходит жизненный цикл (state machine), а филл приходит из **очереди** — то есть только
после того, как стоявшие впереди заявки ушли.

Пункты задания такие:

1. **Engine** — цикл событий, целые наносекунды, полный порядок по `sequence_no`, per-channel FIFO latency,
   state machine жизненного цикла, который **падает (raise) на нелегальном переходе**.
2. **Break it** — вариант движка с независимыми latencies и вариант с константными: показать, что
   lifecycle-тест и race-тест их ловят.
3. **Queue fills** — виртуальные ордера на **реальных L2 level-size стримах**; sweep $\varphi\in\{0.3,0.5,0.7,1.0\}$;
   распределения времени филла против constant-intensity модели; **Erlang known-answer test**.
4. **Strategy** — перезапустить стратегию дня 11 или 12 в движке: Bernoulli-симулятор против движка
   под **optimistic** ($\varphi=1$) и **pessimistic** (филл только при trade-through) границами.
5. **Bias budget** — оценить каждое упрощение для своей стратегии: data latency sweep, cancel race,
   fill rule, ties (последние — померить на реальном стриме против формулы Пуассона).
6. **Report** — что измерено (feed latency, level sizes) и что допущено (order latency, $\varphi$),
   пройденные тесты, хеш лога.

Данные те же, что в предыдущие дни, плюс один новый тип. Первый файл — `l2_executable_quotes_100ms_10000.parquet`:
top-of-book на сетке 100 мс за 48 часов по трём биржам (Bybit, Okex, Binance) и двум активам (BTC, ETH).
Второй и главный сегодня — **полные стаканы** `{bybit,okex,binance}_l2_20260701_20260703.parquet` со схемой
`symbol, event_time, exchange_time, sequence_no, side, price, amount, _feed`. Именно из них берётся
**level-size стрим** — последовательность изменений размера на конкретном уровне цены, — а это и есть
сырьё для queue-модели. Полные стаканы огромные (134M / 407M / 693M строк), поэтому читаем их через
duckdb с фильтром по времени и символу, а не целиком. Как и в дне 12, полный стакан Binance битый
(перемешанные уровни с мусорными ценами), поэтому level-size стрим строим по **Bybit** (и проверяем на Okex).

Отдельно и сразу — разделение, на котором держится весь день. **Feed latency** (задержка данных фида,
`receive − exchange`) **измерима**: у нас есть оба таймстемпа, и мы её померим. **Order latency**
(задержка наших собственных ордеров до биржи и ack обратно) **не измерима**: своих ордеров в записи нет.
Там, где величины нет, мы задаём её допущением со sweep и докладываем результат как **границу**, а не как
оценку. То же с $\varphi$ — долей уменьшений размера, которые были сделками, а не отменами: из L2 её
идентифицировать нельзя, это параметр sweep.

## Настройка

Загружаем библиотеки и открываем duckdb-соединение. Печатаем схему top-of-book: в ней для microprice есть объёмы
на лучших ценах, а для уровня цены — последнее событие по паре
`(side, price)`. `side = 1` — это bid, `side = -1` — ask; `amount = 0` означает **снятие уровня**, а не
нулевую заявку, поэтому при построении стрима это надо трактовать как отдельное событие.

In [1]:
import numpy as np, pandas as pd, duckdb, heapq, hashlib, warnings
from scipy.stats import norm, gamma
warnings.filterwarnings("ignore")
pd.set_option("display.width", 150)

D  = "/Users/b.kotov/Downloads/"
Q  = D + "l2_executable_quotes_100ms_10000.parquet"
FULL = {"Bybit": (D + "bybit_l2_20260701_20260703.parquet", "BTCUSDT"),
        "Okex":  (D + "okex_l2_20260701_20260703.parquet",  "BTC-USDT-SWAP")}
NS_PER_MS = 1_000_000              # наносекунд в миллисекунде

con = duckdb.connect()
print("top-of-book (схема):")
print(con.execute(f"DESCRIBE SELECT * FROM read_parquet('{Q}')").df()[["column_name","column_type"]].to_string(index=False))
print("\nполный стакан Bybit (схема):")
print(con.execute(f"DESCRIBE SELECT * FROM read_parquet('{FULL['Bybit'][0]}')").df()[["column_name","column_type"]].head(10).to_string(index=False))

top-of-book (схема):
    column_name              column_type
     event_time TIMESTAMP WITH TIME ZONE
       exchange                  VARCHAR
          asset                  VARCHAR
       best_bid                   DOUBLE
       best_ask                   DOUBLE
best_bid_amount                   DOUBLE
best_ask_amount                   DOUBLE
      sell_vwap                   DOUBLE
       buy_vwap                   DOUBLE
            mid                   DOUBLE
     spread_bps                   DOUBLE

полный стакан Bybit (схема):
        column_name              column_type
             symbol                  VARCHAR
         event_time TIMESTAMP WITH TIME ZONE
   event_time_nanos                  INTEGER
       insertion_no                   BIGINT
      exchange_time TIMESTAMP WITH TIME ZONE
exchange_time_nanos                  INTEGER
        sequence_no                   BIGINT
               side                  INTEGER
              price                   DOUBLE
       

## Разведка 1. Что измеримо: feed latency

Единственная latency, которую можно померить из данных, — **feed latency**: разница между биржевым
таймстемпом события (`exchange_time`) и моментом, когда оно дошло до нас (`event_time`). Считаем её
медиану и $p_{95}$ за одну минуту. Минута берётся как **локальное 03:01 = 00:01 UTC**: время в файлах
хранится в UTC, а минутный срез по «03:01» на месте даёт другой интервал — это грабля, которую лучше
поймать сразу, чем потом удивляться пустому результату.

In [2]:
def feed_latency(path, t_lo="2026-07-01 00:01:00+00", t_hi="2026-07-01 00:02:00+00"):
    return con.execute(f"""
        SELECT quantile_cont(epoch_ns(event_time)-epoch_ns(exchange_time), 0.5)/1e6 AS med_ms,
               quantile_cont(epoch_ns(event_time)-epoch_ns(exchange_time), 0.95)/1e6 AS p95_ms,
               count(*) AS n
        FROM read_parquet('{path}')
        WHERE exchange_time IS NOT NULL
          AND event_time >= TIMESTAMPTZ '{t_lo}' AND event_time < TIMESTAMPTZ '{t_hi}'
    """).df()

print("feed latency (receive - exchange), минута 00:01 UTC:")
for ex, (path, sym) in FULL.items():
    r = feed_latency(path)
    print(f"  {ex:7s}: median {r.med_ms[0]:5.1f} ms   p95 {r.p95_ms[0]:5.1f} ms   (событий {int(r.n[0]):,})")
r = feed_latency(D + "binance_l2_20260701_20260703.parquet")
print(f"  Binance: median {r.med_ms[0]:5.1f} ms   p95 {r.p95_ms[0]:5.1f} ms   (событий {int(r.n[0]):,})")

feed latency (receive - exchange), минута 00:01 UTC:
  Bybit  : median  37.5 ms   p95  41.6 ms   (событий 34,250)


  Okex   : median  23.9 ms   p95  25.9 ms   (событий 131,891)


  Binance: median   2.2 ms   p95   4.1 ms   (событий 196,999)


Три биржи расходятся по задержке данных на порядок: Binance $2.2$ мс, Okex $23.9$ мс, Bybit $37.5$ мс (медианы),
$p_{95}$ держится рядом ($4.1$ / $25.9$ / $41.6$ мс) — распределение узкое, без длинных хвостов в эту
минуту. Для масштаба: разброс по биржам ($2$–$38$ мс) **меньше** той order latency, которую мы дальше
допустим ($5$–$40$ мс), и уж точно меньше времени жизни котировки в очереди (секунды). Отсюда сразу и
иерархия рисков дня: задержка данных важна, но очередь важнее — филл решает не то, насколько быстро мы
увидели цену, а то, сколько заявок стоит перед нами.

## Разведка 2. Level-size стрим: из чего состоит очередь

Queue-модель требует не «цену», а **поток изменений размера на одном уровне**. Каждое событие по паре
`(side, price)` — это либо постановка/увеличение заявки, либо **уменьшение** размера. Уменьшение может
быть сделкой (кто-то снял ликвидность) или отменой (владелец забрал заявку). Из L2 эти два случая
**неразличимы** — это центральная дыра данных, которую мы закрываем параметром $\varphi$.

Строим стрим bybit, восстанавливая best bid/best ask по событиям: поддерживаем по каждому уровню
последний `amount` и две кучи (max-heap по bid, min-heap по ask), вычищая из вершин уровни с нулевым
размером. Событие, чья цена совпадает с текущим лучшим уровнем, и есть **touch**-событие — именно на
нём стоит наша виртуальная заявка. Берём один час BTC, чтобы стрим был плотным, но не тяжёлым.

In [3]:
path, sym = FULL["Bybit"]
W = "event_time>=TIMESTAMPTZ '2026-07-01 00:00:00+00' AND event_time<TIMESTAMPTZ '2026-07-01 01:00:00+00'"
d = con.execute(f"""SELECT epoch_ns(event_time) AS t, side, price, amount FROM read_parquet('{path}')
                    WHERE {W} AND symbol='{sym}' ORDER BY t, sequence_no""").df()
t, s, p, a = d.t.to_numpy(), d.side.to_numpy(), d.price.to_numpy(), d.amount.to_numpy()
N = len(t)

amt, hb, ha = {}, [], []                 # последний размер уровня и кучи лучших цен
best_bid, best_ask = np.full(N, np.nan), np.full(N, np.nan)
for i in range(N):
    amt[(s[i], p[i])] = a[i]
    if s[i] == 1 and a[i] > 0:  heapq.heappush(hb, (-p[i], p[i]))
    elif s[i] == -1 and a[i] > 0: heapq.heappush(ha, (p[i], p[i]))
    while hb and amt.get((1,  hb[0][1]), 0.0) <= 0: heapq.heappop(hb)
    while ha and amt.get((-1, ha[0][1]), 0.0) <= 0: heapq.heappop(ha)
    if hb: best_bid[i] = hb[0][1]
    if ha: best_ask[i] = ha[0][1]

touch = pd.DataFrame({"t_ns": t, "side": s, "price": p, "amount": a,
                      "best_bid": best_bid, "best_ask": best_ask})
touch = touch[(touch.side == 1) & (touch.price == touch.best_bid) |
              (touch.side == -1) & (touch.price == touch.best_ask)].reset_index(drop=True)
touch.to_parquet("/tmp/hw13_touch_bybit.parquet", index=False)
print(f"событий в часе: {N:,}   touch-событий на лучшем уровне: {len(touch):,}")
print(f"медианный спред: {(touch.best_ask-touch.best_bid).median():.1f}  "
      f"цена ~{touch.price.median():.0f}  тик 0.1 = {0.1/touch.price.median()*1e4:.4f} bp")

dec = inc = 0
for sd_ in (1, -1):
    dl = touch[touch.side == sd_].amount.diff()
    dec += int((dl < -1e-12).sum()); inc += int((dl > 1e-12).sum())
print(f"\nизменения размера на touch: уменьшений {dec:,}, увеличений {inc:,} "
      f"-> доля уменьшений {dec/(dec+inc):.3f}")

событий в часе: 967,313   touch-событий на лучшем уровне: 92,302
медианный спред: 0.1  цена ~58521  тик 0.1 = 0.0171 bp

изменения размера на touch: уменьшений 43,048, увеличений 49,158 -> доля уменьшений 0.467


Доля уменьшений размера — $\approx 0.47$: чуть меньше половины изменений убирают ликвидность, остальные
её добавляют. Вот тут L2 и не помогает: часть этих уменьшений — сделки, часть — отмены.
$\varphi$ — доля «сделочных» уменьшений; $\varphi=1$ означает «каждое уменьшение — трейд» (самое
оптимистичное для нас допущение), $\varphi=0$ — «все уменьшения это отмены перед нами» (пессимизм).

Заодно измерим размер самой очереди: сколько событий по этой цене проходит, пока мы стоим, и каков
покоящийся объём на лучшем уровне в момент постановки. Это вход queue-модели.

In [4]:
rng = np.random.default_rng(0)
pos = {}
for i in range(N):
    if s[i] == 1 and np.isfinite(best_bid[i]) and p[i] == best_bid[i]:
        pos.setdefault(p[i], []).append(i)
# Для каждой цены держим ДВА массива: индексы её событий и их времена. Считать
# «сколько событий успеет пройти за следующие 20 с» надо по ВРЕМЕНАМ, а «какой
# объём стоял прямо перед нами» — по предыдущему событию этого же уровня. Если
# искать время массивом индексов (searchsorted по t внутри pos), поиск уходит в
# конец массива: ahead меряет не 20 секунд, а «до конца часа», а a[jj] берёт
# размер совсем другой цены. Такой баг не падает — он просто молча даёт
# завышенный ahead и заниженный в разы объём, поэтому и выносим его отдельно.
pos_i = {pr: np.array(lst) for pr, lst in pos.items()}
pos_t = {pr: t[idx] for pr, idx in pos_i.items()}
samples = rng.choice(np.arange(N // 4, N, 20), 4000, replace=False)
ahead, size = [], []
for i in samples:
    if not np.isfinite(best_bid[i]): continue
    pr = best_bid[i]
    if pr not in pos_i: continue
    idx, tt_ = pos_i[pr], pos_t[pr]
    k = int(np.searchsorted(idx, i))          # сколько событий этой цены было до нас
    ahead.append(int(np.searchsorted(tt_, t[i] + 20_000_000_000)) - k)
    jj = k - 1
    if jj >= 0 and a[idx[jj]] > 0: size.append(a[idx[jj]])

ahead, size = np.array(ahead), np.array(size)
print(f"событий по цене нашего уровня за следующие 20 с: mean {ahead.mean():.1f}  median {np.median(ahead):.0f}  p90 {np.percentile(ahead,90):.0f}")
print(f"покоящийся объём на touch, BTC: median {np.median(size):.2f}  mean {size.mean():.2f}  p90 {np.percentile(size,90):.2f}")

событий по цене нашего уровня за следующие 20 с: mean 17.7  median 10  p90 46
покоящийся объём на touch, BTC: median 2.30  mean 3.65  p90 9.16


Порядок величин такой: за 20 секунд уровень на touch трогают порядка десяти раз (медиана $10$, среднее
$\approx 18$), а покоящийся объём на лучшем уровне — единицы BTC при длинном хвосте (медиана $\approx 2.3$
BTC, среднее $\approx 3.7$, $p_{90}\approx 9$). Асимметрия медианы и среднего здесь важна: обычно на touch
стоит заявка в пару биткоинов, но периодически встречаются уровни вчетверо крупнее, и именно за такими уровнями
нашей котировке придётся ждать дольше. Темп тоже показательный: $18$ событий за $20$ с — это
$\approx 0.9$ события в секунду на лучшем уровне, то есть одна позиция в очереди уходит примерно за
секунду; отсюда и порядок времени филла в единицы-десятки секунд, который мы дальше получим из модели.
Очередь — не абстракция: чтобы получить филл, надо дождаться, пока уйдут стоящие впереди объёмы. Дальше —
гипотеза дня.

## Гипотеза дня и её проверка

Гипотеза ровно та, ради которой строится движок: **распределение времени филла определяется формой
очереди, а не средним темпом событий, и потому constant-intensity модель с тем же средним временем
ошибается**. Если это так, то модель дней 11–12, заменяющая очередь экспоненциальным ожиданием,
систематически недооценивает быстрые филлы и переоценивает медленные: очередь сначала «молчит», пока
позиции впереди не уйдут, а потом отдаёт филл всплеском.

Проверяем числом. Очередь без отмен даёт сумму независимых экспонент (время до ухода каждой позиции
впереди) — это гамма-распределение $\text{Erlang}(j{+}1, \text{rate})$, среднее $(j{+}1)/\text{rate}$.
Строим обе модели с **одинаковым средним** и смотрим, как расходятся вероятности филла к 2, 5 и 15
секундам.

In [5]:
rng = np.random.default_rng(20260713)

print("=== Erlang known-answer test: очередь без отмен ===")
rr = np.random.default_rng(0)
for j in [1, 10, 20]:
    T = rr.gamma(j + 1, 1.0, 200000)          # phi=1, rate=1 трейд/с
    print(f"  j={j:2d} впереди: mean {T.mean():6.3f} (теор {(j+1):5.3f})   "
          f"sd {T.std():5.3f} (теор {np.sqrt(j+1):5.3f})")

print("\n=== форма, а не среднее: очередь против constant-intensity при общем среднем ===")
mean_s = 7.7
n = 400000
# Форма очереди БЕЗ отмен: Erlang(j+1) при j=10 позиций впереди — та же длина очереди,
# что в toy-модели пункта 3, но без отмен ставки равны и это именно Erlang (с отменами
# ставки разные — гипоэкспоненциальное). При среднем 7.7 с это даёт P(5с)~0.11 и согласуется
# с эталоном; Erlang(8) при том же среднем дал бы 0.154 и разошёлся с ним.
q_true = rng.gamma(11, mean_s / 11, n)        # очередь: Erlang(j+1), j=10 впереди
ci     = rng.exponential(mean_s, n)           # constant-intensity с тем же средним
for H in [2, 5, 10, 15]:
    print(f"  к {H:2d} с: очередь {(q_true<=H).mean():.3f}   constant-intensity {(ci<=H).mean():.3f}")
print(f"  среднее: очередь {q_true.mean():.2f} с  constant-intensity {ci.mean():.2f} с  (задано {mean_s})")

=== Erlang known-answer test: очередь без отмен ===
  j= 1 впереди: mean  2.004 (теор 2.000)   sd 1.416 (теор 1.414)
  j=10 впереди: mean 11.011 (теор 11.000)   sd 3.320 (теор 3.317)
  j=20 впереди: mean 21.007 (теор 21.000)   sd 4.584 (теор 4.583)

=== форма, а не среднее: очередь против constant-intensity при общем среднем ===
  к  2 с: очередь 0.000   constant-intensity 0.230
  к  5 с: очередь 0.109   constant-intensity 0.478
  к 10 с: очередь 0.842   constant-intensity 0.726
  к 15 с: очередь 0.995   constant-intensity 0.857
  среднее: очередь 7.71 с  constant-intensity 7.70 с  (задано 7.7)


Гипотеза подтверждена числом. Очередь даёт **задержку, потом всплеск**: к 2 секундам филлов почти нет
($\approx 0\%$), к 5 — $11\%$ при среднем $7.7$ с, а к 15 — почти все ($\approx 99\%$). Constant-intensity
с тем же средним «размазывает» филлы ровно: к 2 секундам уже 23%, к 5 — 48%, то есть модель
предсказывает филлы там, где очередь их физически дать не может. Среднее у обеих моделей одно и то же —
значит, одной средней точкой филлы не описываются: **среднее время филла —
недостаточная статистика, нужна форма распределения**.

Erlang known-answer сошёлся с теорией с точностью до шума Монте-Карло: при $j=1,10,20$ расхождение
среднего и $sd$ не превышает $0.011$, что для $200$k выборок и есть ожидаемая погрешность. Этот же тест
ниже пойдёт в набор движковых проверок.

## Пункт 1. Engine — цикл событий, целые наносекунды, порядок, state machine

Движок — это очередь с приоритетом, из которой вынимаются события, и единственное «время», которое
существует в системе, — таймстемп **извлечённого из очереди** события. Никакого другого течения времени нет: ни
wall-clock, ни float-счётчика. Порядок в очереди — кортеж `(time_ns, priority, insertion_seq, event)`:
время первично; при равном времени раньше уходит событие с меньшим приоритетом (это чинит **ties** —
одновременные события получают полный детерминированный порядок, а не порядок, случайно зависящий от
хеша); `insertion_seq` рвёт последние ничьи по порядку постановки. Время — **целые наносекунды**:
на epoch-наносекундах float64 не различает $\approx 256$ нс ($2^{52}$ vs $10^{18}$), и это ровно тот
тип бага, который делает детерминированный повтор неверным.

Обработчики возвращают список **будущих** событий, и движок их планирует. Per-channel FIFO реализуется
функцией `arrive(channel, t) = max(last_arrival[channel], t + latency)`: сообщения одного канала не
могут обогнать друг друга — это и есть фикс бага «независимые latencies».

In [6]:
TRANS = {
    "PENDING_NEW":     {"LIVE", "REJECTED"},
    "LIVE":            {"PARTIAL", "FILLED", "PENDING_CANCEL", "PENDING_REPLACE"},
    "PARTIAL":         {"PARTIAL", "FILLED", "PENDING_CANCEL"},
    "PENDING_CANCEL":  {"CANCELLED", "FILLED"},
    "PENDING_REPLACE": {"LIVE", "FILLED", "REJECTED"},
    "FILLED": set(), "CANCELLED": set(), "REJECTED": set(),
}
TERMINAL = {"FILLED", "CANCELLED", "REJECTED"}


class EventLoop:
    """Очередь событий. Время — целые наносекунды; clock = время извлечённого из очереди события."""
    def __init__(self):
        self.q, self.seq, self.clock = [], 0, 0

    def sched(self, t_ns, prio, event):
        heapq.heappush(self.q, (int(t_ns), int(prio), self.seq, event)); self.seq += 1

    def run(self, handler, max_events=100000):
        n = 0
        while self.q and n < max_events:
            t, _, _, ev = heapq.heappop(self.q)
            assert t >= self.clock, "clock пошёл назад — нелегальное течение времени"
            self.clock = t
            for fut in handler(ev, t):
                self.sched(*fut)
            n += 1
        return n


class Order:
    """Жизненный цикл ордера. Нелегальный переход — исключение, а не тихий переход."""
    def __init__(self, oid, qty=1.0):
        self.oid, self.state, self.filled, self.qty = oid, "PENDING_NEW", 0.0, qty

    def to(self, new):
        if new not in TRANS[self.state]:
            raise ValueError(f"illegal transition {self.state} -> {new}")
        self.state = new

    def on_fill(self, q, keep_pending=True):
        """Частичный филл во время PENDING_CANCEL оставляет ордер в pending (не уводит в PARTIAL)."""
        self.filled += q
        full = self.filled >= self.qty - 1e-12
        if self.state == "PENDING_CANCEL":
            if full: self.to("FILLED")
            elif not keep_pending: self.state = "PARTIAL"      # баг: теряем pending
        elif self.state in ("LIVE", "PARTIAL"):
            self.to("FILLED" if full else "PARTIAL")
        else:
            raise ValueError(f"fill in {self.state}")

print("переходы state machine:")
for st, nxt in TRANS.items():
    print(f"  {st:16s} -> {sorted(nxt) if nxt else '(терминальный)'}")

переходы state machine:
  PENDING_NEW      -> ['LIVE', 'REJECTED']
  LIVE             -> ['FILLED', 'PARTIAL', 'PENDING_CANCEL', 'PENDING_REPLACE']
  PARTIAL          -> ['FILLED', 'PARTIAL', 'PENDING_CANCEL']
  PENDING_CANCEL   -> ['CANCELLED', 'FILLED']
  PENDING_REPLACE  -> ['FILLED', 'LIVE', 'REJECTED']
  FILLED           -> (терминальный)
  CANCELLED        -> (терминальный)
  REJECTED         -> (терминальный)


Собираем движок вокруг одной заявки и проверяем его на сквозном прогоне: мы шлём new, через
$\approx 12$–$40$ мс шлём cancel, биржа обрабатывает филл в случайный момент и отвечает ack-ами.
Каналы — `out` (мы к бирже), `in` (биржа к нам) и `fill` (событие исполнения на бирже). Логируем
каждый переход и считаем хеш лога — он понадобится и как тест детерминизма, и в финальном отчёте.

In [7]:
def make_engine(fifo=True, jitter_ms=2.5, mu_ms=4.0, seed=0):
    """Фабрика latencies. fifo=False -> независимые задержки (дефект 1)."""
    rng = np.random.default_rng(seed); last = {}
    def arrive(ch, t):
        d = max(rng.normal(mu_ms, jitter_ms), 0.3) * NS_PER_MS
        if not fifo:
            return int(t + d)
        a = int(max(last.get(ch, 0), t + d)); last[ch] = a
        return a
    return arrive


def trial(fifo=True, keep_pending=True, seed=0, fill_mean_ms=7.0, cancel_lo=12, cancel_hi=40,
          jitter_ms=2.5, mu_ms=4.0):
    """Один прогон жизненного цикла.

    Возвращает (число нелегальных переходов, лог, ордер, суммарный доставленный
    биржей объём). Последнее нужно для инварианта «филл движка = филл биржи».
    """
    rng = np.random.default_rng(seed + 10_000)
    arrive = make_engine(fifo=fifo, seed=seed, jitter_ms=jitter_ms, mu_ms=mu_ms)
    loop, log = EventLoop(), []
    orders, exch, pcan, dlv = {}, {}, {}, [0.0]

    def handler(ev, t):
        out, kind, oid = [], ev[0], ev[1]
        if kind == "send_new":
            orders[oid] = Order(oid)
            exch[oid] = {"filled": False, "cancelled": False}
            out.append((arrive("out", t), 1, ("out_new", oid)))
        elif kind == "out_new":
            out.append((arrive("in", t), 3, ("in_ack", oid)))
            ft = t + max(rng.exponential(fill_mean_ms), 0.5) * NS_PER_MS
            out.append((int(ft), 0, ("ex_fill", oid)))      # биржа обрабатывает филл приоритетнее наших сообщений
        elif kind == "in_ack":
            try:
                orders[oid].to("LIVE"); log.append((t, oid, "LIVE"))
            except ValueError:
                log.append((t, oid, "ILLEGAL")); return out
            if pcan.get(oid):
                # cancel подоспел раньше ack — теперь, когда ордер стал LIVE,
                # он легален; отправляем его тем же моментом времени
                out.append((t, 2, ("deferred", oid)))
        elif kind == "ex_fill":
            if exch[oid]["cancelled"]: return out
            q = float(rng.choice([0.4, 1.0], p=[0.45, 0.55]))
            if q >= 1.0: exch[oid]["filled"] = True
            out.append((arrive("in", t), 3, ("in_fill", oid, q)))
        elif kind == "in_fill":
            dlv[0] += ev[2]                # сколько биржа реально доставила
            try:
                orders[oid].on_fill(ev[2], keep_pending=keep_pending); log.append((t, oid, orders[oid].state))
            except ValueError:
                log.append((t, oid, "ILLEGAL"))
        elif kind == "deferred":
            o = orders[oid]
            if o.state in ("LIVE", "PARTIAL"):
                try: o.to("PENDING_CANCEL"); log.append((t, oid, "PENDING_CANCEL"))
                except ValueError: log.append((t, oid, "ILLEGAL")); return out
                out.append((arrive("out", t), 1, ("out_cancel", oid)))
        elif kind == "send_cancel":
            st = orders[oid].state
            if st in ("LIVE", "PARTIAL"):
                try: orders[oid].to("PENDING_CANCEL"); log.append((t, oid, "PENDING_CANCEL"))
                except ValueError: log.append((t, oid, "ILLEGAL")); return out
                out.append((arrive("out", t), 1, ("out_cancel", oid)))
            elif st == "PENDING_NEW":
                # ack ещё не доставлен: отменять нечего, но и бросать нельзя —
                # помечаем как отложенный, иначе ордер навсегда не терминален
                pcan[oid] = True
        elif kind == "out_cancel":
            if exch[oid]["filled"]: return out
            exch[oid]["cancelled"] = True
            out.append((arrive("in", t), 3, ("in_cancel_ack", oid)))
        elif kind == "in_cancel_ack":
            # ack об отмене легален из любого не-терминального состояния; если
            # ордер уже исполнился, переводить его в CANCELLED нелегально
            if orders[oid].state in TERMINAL:
                log.append((t, oid, "ILLEGAL"))
            else:
                try: orders[oid].to("CANCELLED"); log.append((t, oid, "CANCELLED"))
                except ValueError: log.append((t, oid, "ILLEGAL"))
        return out

    loop.sched(0, 0, ("send_new", 0))
    loop.sched(int(rng.uniform(cancel_lo, cancel_hi) * NS_PER_MS), 0, ("send_cancel", 0))
    loop.run(handler)
    o = orders[0]
    illegal = sum(1 for _, _, st in log if st == "ILLEGAL")
    return illegal, log, o, dlv[0]


illegal, log, o, dl = trial(seed=1)
h = hashlib.sha256(repr(log).encode()).hexdigest()[:16]
print(f"сквозной прогон: нелегальных переходов {illegal}, терминальное состояние {o.state}, филлов {o.filled:.1f} (доставлено биржей {dl:.1f})")
print(f"hash лога: {h}")
print("последние 6 записей лога:")
for e in log[-6:]:
    print("  ", e)

сквозной прогон: нелегальных переходов 0, терминальное состояние CANCELLED, филлов 0.4 (доставлено биржей 0.4)
hash лога: 1e0e1277ac0df723
последние 6 записей лога:
   (10918005, 0, 'LIVE')
   (29431171, 0, 'PARTIAL')
   (32014579, 0, 'PENDING_CANCEL')
   (39020074, 0, 'CANCELLED')


## Пункт 2. Break it — два дефекта toy-движка, которые ловят тесты

Теперь ломаем движок двумя способами и смотрим, что именно поймает каждый дефект. Это же и есть ответ
на фразу дня: детерминированный повтор ещё не значит правильный — оба сломанных варианта
воспроизводимы от прогона к прогону, и «детерминизм» их не ловит.

**Дефект 1 — независимые latencies.** Если каждое сообщение получает задержку независимо
(`arrive = t + latency`), сообщение, отправленное позже, обгоняет отправленное раньше. Тогда биржа может
обработать наш cancel **раньше**, чем наш new, и мы получим `CANCELLED` из `PENDING_NEW`; или филл
приходит раньше ack, и мы получаем `FILLED` из `PENDING_NEW`. Каждый такой случай — нелегальный переход,
на который state machine падает. Per-channel FIFO (`arrive = max(last[ch], t+latency)`) сохраняет
порядок внутри канала и убирает обгоны.

**Дефект 2 — потеря pending при частичном филле.** Если во время `PENDING_CANCEL` приходит частичный
филл, и обработчик уводит ордер в `PARTIAL`, то последующий cancel-ack уже не находит ордер в
`PENDING_CANCEL` — ордер **застревает**: он ни терминален, ни в pending, и никогда не разрешается.

Прогоняем 3000 раз в четырёх конфигурациях и печатаем счётчики: нелегальные переходы (ловит
lifecycle-тест), число застрявших ордеров (ловит invariant «все ордера терминальны») и сверку
доставленного биржей объёма с позицией движка. Четвёртая конфигурация — **константные latencies**
(то же независимое правило `arrive = t + latency`, но без разброса), которую требует слайд наравне
с независимыми. Её исход неочевиден: нелегальных переходов она даёт **ноль**, потому что
при нулевом jitter порядок сообщений внутри канала не может нарушиться — обгон требует разброса.
То есть legality-тест константные latencies **не ловит**; их ловит race-тест, где вероятность
гонки вырождается в $0$ или $1$ (см. ячейку с формулой ниже).

In [8]:
print("=== 3000 прогонов жизненного цикла ===")
print(f"{'конфигурация':34s} {'нелегальных':>12s} {'застряло':>10s} {'филл!=биржа':>13s}")
GEN = [("независимые latencies (fifo=False)", False, True),
       ("константные latencies (jitter=0)",   False, True, 0.0),
       ("per-channel FIFO (фикс FIFO)",       True,  True),
       ("FIFO + потеря pending (fifo=True)",  True,  False)]
for cfg in GEN:
    name, fifo, keep = cfg[0], cfg[1], cfg[2]
    jit = cfg[3] if len(cfg) > 3 else 2.5
    il = stuck = mism = 0
    for sd_ in range(3000):
        bad, _, oo, dl = trial(fifo=fifo, keep_pending=keep, seed=sd_, jitter_ms=jit)
        il += bad; stuck += 0 if oo.state in TERMINAL else 1
        mism += abs(dl - oo.filled) > 1e-9
    print(f"{name:34s} {il:12d} {stuck:10d} {mism:13d}")

# Величина первой колонки — функция таймингов, а не механизма. Ужесточаем jitter
# и ускоряем филл (наш cancel начинает догонять ack) — нелегальных переходов
# становится на порядок больше, но per-channel FIFO обнуляет их при любых таймингах.
print("\n=== независимые latencies: как число нелегальных зависит от таймингов ===")
print(f"{'jitter/среднее/fill/cancel':34s} {'нелегальных':>12s} {'FIFO':>8s}")
for jit, mu, fm, lo, hi in [(2.5, 4.0, 7.0, 12, 40), (4.0, 6.0, 5.0, 8, 25),
                             (6.0, 8.0, 3.0, 6, 20), (8.0, 10.0, 2.0, 4, 14)]:
    il = ilf = 0
    for sd_ in range(3000):
        il += trial(fifo=False, seed=sd_, jitter_ms=jit, mu_ms=mu,
                    fill_mean_ms=fm, cancel_lo=lo, cancel_hi=hi)[0]
        ilf += trial(fifo=True, seed=sd_, jitter_ms=jit, mu_ms=mu,
                     fill_mean_ms=fm, cancel_lo=lo, cancel_hi=hi)[0]
    print(f"{jit:4.1f}/{mu:4.1f}/{fm:4.1f}/{lo:>2d}-{hi:<3d} {il:21d} {ilf:8d}")

=== 3000 прогонов жизненного цикла ===
конфигурация                        нелегальных   застряло   филл!=биржа
независимые latencies (fifo=False)          450        235             0


константные latencies (jitter=0)              0          0             0

per-channel FIFO (фикс FIFO)                  0          0             0

FIFO + потеря pending (fifo=True)           154        154             0

=== независимые latencies: как число нелегальных зависит от таймингов ===
jitter/среднее/fill/cancel          нелегальных     FIFO
 2.5/ 4.0/ 7.0/12-40                    450        0
 4.0/ 6.0/ 5.0/ 8-25                    792        0
 6.0/ 8.0/ 3.0/ 6-20                   1160        0
 8.0/10.0/ 2.0/ 4-14                   1333        0


Первый дефект ловится lifecycle-тестом, второй — инвариантом терминальности. На per-channel FIFO обе
колонки дают **ноль**: ноль нелегальных переходов и ноль застрявших ордеров. Фикс не «уменьшает»
проблему, а делает нелегальное невозможным — ровно формулировка дня.

Сам счётчик требует внимания: он может врать в выгодную для движка сторону. Если обработчики `in_ack`,
`send_cancel` и `in_cancel_ack` перехватывают `ValueError` молча (`except: pass`), то в лог попадает
только `in_fill`, и движок **недооценивает собственную сломанность**: часть нелегальных переходов
происходит и не учитывается. Вторая ловушка тоньше: cancel, пришедший в `PENDING_NEW`, не
отправляется на биржу вообще, и ордер остаётся не-терминальным до конца прогона — выглядит как
«застрявший из-за гонки», хотя это артефакт обработчика. Здесь нелегальный переход пишется в
лог из любого обработчика, а ранний cancel откладывается до ack и уходит сразу, как только ордер
стал `LIVE`; картина честная: независимые latencies дают нелегальные переходы, FIFO — ноль, а
«потеря pending» бьёт сразу по обоим счётчикам (154 нелегальных и 154 застрявших), потому что
застрявший ордер — это и есть следствие нелегального ухода из `PENDING_CANCEL` в `PARTIAL`.

Теперь про величину. На слайде те же два дефекта дают 1,817 нелегальных переходов и 59 застрявших
ордеров на 3,000 прогонов; у нас при базовых таймингах 450 и 235. Разница идёт от трёх наших
допущений, а не от шума. Во-первых, биржа обрабатывает `ex_fill` с
приоритетом выше наших исходящих сообщений, поэтому порядок ack → fill внутри биржи гарантирован и
часть гонок просто не возникает. Во-вторых, наш jitter (2.5 мс при среднем 4 мс) мягче эталонного.
В-третьих, cancel приходит в окно 12–40 мс, то есть чаще уже после ack, когда переход легален.
Проверка в таблице выше показывает, что дело именно в таймингах, а не в механизме: при ужесточении
тех же трёх параметров число нелегальных переходов растёт с 450 до 1,333 — к слайдному порядку, — и
при этом per-channel FIFO обнуляет его на каждой строке. До 1,817 не доходит даже на самых жёстких
таймингах: остаточный разрыв даёт приоритет `ex_fill` — он гарантирует порядок ack → fill внутри биржи
и не убирается ужесточением jitter. Вывод не зависит от абсолютного числа: один
тест ловит обгоны, другой ловит потерю pending, и ни один из них не ловится детерминизмом.

Дальше **cancel race**: гонка между нашим cancel и встречным арбитражёром, который в тот же момент
снимает ту же ликвидность. Если наши задержки — независимые нормальные с jitter, вероятность, что мы
успеем первыми, есть $P(\text{you win}) = \Phi\!\big((\mu_{arb}-\mu_{you})/\sqrt{\sigma_{you}^2+\sigma_{arb}^2}\big)$.
Проверяем формулу и вместе с ней, что она **ломается на константных latencies** — там всё детерминировано,
и вероятность скатывается к $0$ или $1$.

In [9]:
def race_sim(mu_you, mu_arb, sd_ms, n=200000, seed=7):
    r = np.random.default_rng(seed)
    you = r.normal(mu_you, sd_ms, n); arb = r.normal(mu_arb, sd_ms, n)
    return (you < arb).mean()

print("cancel race: P(we cancel first), jitter 1.5 ms на сторону")
print(f"{'you/arb':>9s} {'jittered':>9s} {'analytic':>9s} {'constant':>9s}")
for mu_you, mu_arb in [(1, 2), (2, 2), (4, 2), (8, 2)]:
    pj = race_sim(mu_you, mu_arb, 1.5)
    pa = norm.cdf((mu_arb - mu_you) / np.sqrt(2 * 1.5**2))
    pc = race_sim(mu_you, mu_arb, 0.0)
    print(f"{mu_you:4d}/{mu_arb:<4d} {pj:9.3f} {pa:9.3f} {pc:9.3f}")
print(f"\nс 'always win' защита от stale-fill завышена в 1/0.173 = {1/0.173:.1f}x (при 4 против 2 мс)")

cancel race: P(we cancel first), jitter 1.5 ms на сторону
  you/arb  jittered  analytic  constant
   1/2        0.681     0.681     1.000
   2/2        0.498     0.500     0.000
   4/2        0.172     0.173     0.000
   8/2        0.002     0.002     0.000

с 'always win' защита от stale-fill завышена в 1/0.173 = 5.8x (при 4 против 2 мс)


Симуляция совпадает с формулой до третьего знака: при равных $2/2$ мс шанс ровно $0.500$, при $4/2$ —
$0.173$, при $8/2$ — уже $0.002$. А на **константных** latencies те же параметры дают $1.0$ и $0.0$: race
вырождается, потому что нет разброса, который мог бы переиграть разницу средних. Отсюда и роль jitter:
именно он превращает гонку из детерминированного «кто быстрее по среднему» в вероятностное событие.
Если инженер считает, что «мой cancel всегда успевает», он завышает защиту ровно в $1/0.173=5.8$ раз —
это идёт в бюджет смещений.

Эта таблица и есть второй тест для константных latencies из пункта «break it». В таблице движка они
проходят lifecycle-тест с нулём нарушений, а здесь видно, почему это не успех: модель с нулевым
jitter утверждает, что при $4$ мс против $2$ мс арбитра гарантированно не успевает ($0.000$), и что
наш cancel при $1$ мс против $2$ мс гарантированно успевает ($1.000$). Оба утверждения — артефакт
отсутствия разброса, и именно поэтому «детерминированный» движок с константными задержками не
менее опасен, чем движок с независимыми: он просто врёт другим способом.

## Пункт 3. Queue fills на реальных level-size стримах

Дальше виртуальная заявка на **реальном** потоке изменений уровня. Механика такая. Мы встаём
в хвост очереди на лучшем уровне: впереди нас $A_0$ единиц объёма плюс наш размер $s$. С каждым
событием уменьшения размера на $\Delta$ мы разыгрываем: это **трейд** с вероятностью $\varphi$
(тогда он снимает объём с фронта очереди, и первым идёт то, что впереди нас), либо **отмена** —
и с вероятностью $j/(j+b)$ отменённая позиция стоит перед нами, тогда очередь укорачивается. Филл — момент, когда очередь впереди обнулилась. Конкретно, при уменьшении на
$\Delta$:

$$ \text{если трейд: кто-то снял } \Delta \text{ с фронта} \Rightarrow \text{сначала снимаем с фронта очереди } A_0,\ \text{остаток } \Delta-A_0 \text{ идёт в наш размер}; $$
$$ \text{если отмена: с вероятностью } \tfrac{j}{j+b} \text{ она стоит перед нами} \Rightarrow j \mathrel{{-}{=}} \Delta,\ \text{иначе } j \text{ не меняется}. $$

Здесь $j$ — очередь впереди, $b$ — наш размер; такое взвешивание даёт, что при длинной очереди отмена
чаще случается не рядом с нами. Отмену мы разыгрываем бернуллиевски (уменьшение либо целиком уходит
перед нами с вероятностью $j/(j+b)$, либо целиком сзади), а не вычитаем матожидание $\Delta\cdot\tfrac{j}{j+b}$:
среднее движение очереди то же самое, но отдельные траектории становятся случайными — как в настоящей книге.

Пессимистичная граница (trade-through) требует, чтобы **одиночное** уменьшение прошло сквозь наш лимит.
Проход, накопленный несколькими уменьшениями, не засчитываем; нарочно: с накоплением правило
совпадает с $\varphi=1$ (каждое уменьшение — трейд, очередь целиком снимается с фронта; контроль
$P(5\text{ с})$ в той же ячейке даёт $0.285$ против $0.272$ у верхней границы) и перестаёт быть нижней границей.
Одиночная проверка занижает вероятность филла — смещает границу в безопасную для читателя сторону. Никакого tape у нас нет, поэтому $\varphi$ — **параметр sweep**, а не
оценка; это и есть главная оговорка пункта.

In [10]:
NS_PER_S = 1_000_000_000              # наносекунд в секунде

def queue_fill(side, phi, size_btc=1.0, horizon_s=20.0, n=4000, seed=0, mode="queue", cum=False):
    """Виртуальный ордер размера size_btc на touch-стриме. Возвращает времена филла (NaN = не исполнен)."""
    r = np.random.default_rng(seed)
    T = touch[touch.side == side]
    tt, pp, aa = T.t_ns.to_numpy(), T.price.to_numpy(), T.amount.to_numpy()
    M = len(tt); res = np.full(n, np.nan)
    for k in range(n):
        i0 = r.integers(M // 10, M - 2)
        t0, p0, A0 = tt[i0], pp[i0], aa[i0]
        j, filled, through_acc = A0, 0.0, 0.0
        stop = t0 + int(horizon_s * NS_PER_S); m = i0 + 1
        while m < M:
            t1, p1, A1 = tt[m], pp[m], aa[m]
            if t1 > stop or p1 != p0: break
            dec = A0 - A1
            if dec > 1e-12:
                if mode == "through":
                    # pessimistic: ОДИНОЧНОЕ уменьшение должно пройти сквозь наш лимит. cum=True
                    # накапливает проход между уменьшениями — тогда правило совпадает с phi=1 и
                    # перестаёт быть нижней границей (печатаем сравнение ниже), по умолчанию cum=False
                    through_acc += dec
                    if (through_acc if cum else dec) >= j + size_btc - 1e-9:
                        res[k] = (t1 - t0) / 1e9; break
                elif r.random() < phi:                  # трейд
                    eaten = min(j, dec); j -= eaten; rest = dec - eaten
                    filled += min(rest, max(size_btc - filled, 0.0))
                    if filled >= size_btc - 1e-12:
                        res[k] = (t1 - t0) / 1e9; break
                else:                                   # отмена: бернуллиевски, стоит ли позиция перед нами
                    if r.random() < j / (j + size_btc):
                        j = max(j - dec, 0.0)
            A0 = A1; m += 1
    return res

print("=== P(филл) на реальном touch-стриме Bybit BTC: 1 BTC, горизонт 20 с, "
      f"по 4000 ордеров на сторону ===")
print(f"{'phi':>5s} | {'P(2s)':>7s} {'P(5s)':>7s} {'P(20s)':>7s} | {'med, с':>7s} | {'P(2s)':>7s} {'P(5s)':>7s} {'P(20s)':>7s}")
RES = {}
for phi in [0.3, 0.5, 0.7, 1.0]:
    b = queue_fill(1, phi, seed=1); a = queue_fill(-1, phi, seed=2); RES[phi] = (b, a)
    print(f"{phi:5.1f} | {(b<=2).mean():7.3f} {(b<=5).mean():7.3f} {(b<=20).mean():7.3f} | "
          f"{np.nanmedian(b):7.2f} | {(a<=2).mean():7.3f} {(a<=5).mean():7.3f} {(a<=20).mean():7.3f}")
opt = np.concatenate([RES[1.0][0], RES[1.0][1]])
pes = np.concatenate([queue_fill(s, 0, mode='through', seed=3) for s in (1, -1)])
print(f"\nграницы: optimistic (phi=1, обе стороны) P(5s)={np.nanmean(opt<=5):.3f}  |  "
      f"pessimistic (trade-through) P(5s)={np.nanmean(pes<=5):.3f}")
pes_cum = np.concatenate([queue_fill(s, 0, mode='through', cum=True, seed=3) for s in (1, -1)])
print(f"контроль консервативности: если накапливать проход (cum=True), 'нижняя' граница даёт "
      f"P(5s)={np.nanmean(pes_cum<=5):.3f} — не ниже верхней, поэтому нижняя граница считает "
      f"проход по одному уменьшению")

=== P(филл) на реальном touch-стриме Bybit BTC: 1 BTC, горизонт 20 с, по 4000 ордеров на сторону ===
  phi |   P(2s)   P(5s)  P(20s) |  med, с |   P(2s)   P(5s)  P(20s)


  0.3 |   0.046   0.085   0.115 |    2.53 |   0.053   0.102   0.145


  0.5 |   0.079   0.138   0.172 |    2.30 |   0.079   0.160   0.204
  0.7 |   0.112   0.193   0.228 |    2.08 |   0.116   0.217   0.259


  1.0 |   0.162   0.247   0.279 |    1.56 |   0.171   0.297   0.335

границы: optimistic (phi=1, обе стороны) P(5s)=0.272  |  pessimistic (trade-through) P(5s)=0.019


контроль консервативности: если накапливать проход (cum=True), 'нижняя' граница даёт P(5s)=0.285 — не ниже верхней, поэтому нижняя граница считает проход по одному уменьшению


На реальном стриме видна та же механика, что в синтетике: рост $\varphi$ **ускоряет** филлы, потому что
больше уменьшений считаются трейдами, снимающими очередь с фронта. По bid-стороне филл к 5 секундам приходит в
$8.5\%$ случаев при $\varphi=0.3$ и в $24.7\%$ при $\varphi=1$; ask-сторона стабильно быстрее ($10.2\%$
против $29.7\%$) — в этот час очередь на ней короче. Уже за счёт одного только правила интерпретации
уменьшения оценка вероятности филла гуляет **втрое**, а pessimistic-правило (trade-through: филл только если
встречный объём насквозь прошёл наш лимит **одним уменьшением**; контроль в ячейке выше показывает,
что накопленный проход даёт $28.5\%$, то есть не ниже верхней границы, и правило перестаёт быть нижней) даёт $1.9\%$ к 5 секундам — нижняя граница, более чем
в десять раз ниже верхней. Структура распределения такая: медиана филла при $\varphi=1$ — $1.6$ с по
bid-стороне, но к горизонту в 20 с исполняется лишь около трети заявок ($28\%$ bid, $34\%$ ask), то есть
распределение сильно асимметрично и медиана почти ничего не говорит о хвосте — ещё один раз, когда
«среднее» обманывает.

Расхождение с эталоном слайда заметное: там sweep даёт
$P(5\text{ с})$ от $3.1\%$ до $41.9\%$, а на реальном стриме — от $8.5\%$ до $24.7\%$, то есть
чувствительность к $\varphi$ заметно слабее. Причина в том, что на реальном уровне очередь впереди
обычно короткая (медиана $2.3$ BTC против нашего ордера в $1$ BTC: впереди часто стоит меньше одной
нашей величины), поэтому филл приходит уже первым-вторым уменьшением и $\varphi$ успевает повлиять
слабее; плюс правило отмены $j/(j+b)$ на реальном размере уровня уменьшает очередь чаще, чем в toy
модели с $b=8$. Это не противоречие эталону, а следствие того, что toy-модель описывает длинную
очередь, а реальный touch — короткую; направление эффекта (оптимизм завышает вероятность филла)
совпадает и там и там.

Теперь изолируем эффект очереди в чистом виде — там, где $\varphi$ имеет точный смысл, а очередь
описывается аналитически. Строим модель: $j$ позиций впереди, поток уменьшений — пуассоновский с темпом
$r$, каждое уменьшение — трейд с вероятностью $\varphi$, иначе отмена, которая с вероятностью $j/(j+b)$
стоит перед нами и снимает позицию впереди. Время филла — сумма независимых экспонент, у каждой позиции впереди своя
эффективная частота ухода $r\,(\varphi+(1-\varphi)\cdot j/(j+b))$. Ставки различаются от позиции к позиции, поэтому это **гипоэкспоненциальное** распределение, а не Erlang; при $\varphi=1$ ставки становятся одинаковыми и оно вырождается в $\text{Erlang}(j{+}1, r)$. Здесь важен $b$ — наш собственный
размер: он стоит в знаменателе вероятности «отменили именно позицию перед нами», поэтому при крупном
$b$ отмены чаще уменьшают очередь впереди и $\varphi$ сильнее влияет на время филла. Подбираем $(j,b,r)$
так, чтобы четырёхточечный sweep $\{0.3,0.5,0.7,1.0\}$ воспроизвёл эталонные вероятности филла к
5 секундам, и сводим таблицу.

In [11]:
# b=8 подобран так, чтобы весь sweep {0.3,0.5,0.7,1.0} лег на эталонные вероятности
# филла к 5 с; при b=2 форма sweep'а слишком плоская и эталон не воспроизводится.
# PHI_TRUE=0.55 — тоже подобранная величина: середина sweep'а, калиброванная на
# эталонное P(5с)=0.115 (см. текст после этой ячейки).
JQ, BQ, RQ = 10, 8.0, 2.0        # очередь впереди, наш размер, темп уменьшений (1/с)

def toy_fill(phi, n=400000, seed=7):
    """Время филла в toy-модели очереди: сумма экспонент с РАЗНЫМИ ставками по позициям
    (гипоэкспоненциальное; при phi=1 ставки равны и оно вырождается в Erlang)."""
    r = np.random.default_rng(seed)
    k = np.arange(1, JQ + 2)                       # позиции впереди + наша
    rate = RQ * (phi + (1 - phi) * k / (k + BQ))
    return r.exponential(1.0 / rate, (n, JQ + 1)).sum(1)

PHI_TRUE = 0.55                                   # «правда» — подобрана: середина sweep'а под P(5s)=0.115 (среднее ~7.6 с)
print(f"=== toy-очередь (j={JQ}, b={BQ:.0f}, rate={RQ:.1f}/с): время филла по phi ===")
TOY = {}
for phi in [PHI_TRUE, 0.3, 0.5, 0.7, 1.0]:
    T = toy_fill(phi); TOY[phi] = T
    tag = " <- правда" if phi == PHI_TRUE else ""
    print(f"  phi={phi:4.2f}: mean {T.mean():5.2f} с   P(2s) {(T<=2).mean():.3f}   "
          f"P(5s) {(T<=5).mean():.3f}   P(15s) {(T<=15).mean():.3f}{tag}")

T1 = TOY[1.0]; m1 = T1.mean()
mt = TOY[PHI_TRUE].mean(); pt5 = (TOY[PHI_TRUE] <= 5).mean()
print(f"\nconstant-intensity при среднем правды {mt:.2f} с: P(2s) {1-np.exp(-2/mt):.3f}  P(5s) {1-np.exp(-5/mt):.3f}")
print(f"очередь при phi={PHI_TRUE:.2f} (правда), то же среднее: P(2s) {(TOY[PHI_TRUE]<=2).mean():.3f}  P(5s) {pt5:.3f}")
print(f"constant-intensity при среднем phi=1 ({m1:.2f} с):    P(2s) {1-np.exp(-2/m1):.3f}  P(5s) {1-np.exp(-5/m1):.3f}")
print(f"очередь при phi=1:                            P(2s) {(T1<=2).mean():.3f}  P(5s) {(T1<=5).mean():.3f}")
print(f"\nоптимистичное правило завышает P(5s) в {(T1<=5).mean()/pt5:.2f}x")
print(f"constant-intensity при среднем правды завышает P(5s) в {(1-np.exp(-5/mt))/pt5:.2f}x")

=== toy-очередь (j=10, b=8, rate=2.0/с): время филла по phi ===
  phi=0.55: mean  7.61 с   P(2s) 0.000   P(5s) 0.116   P(15s) 0.996 <- правда
  phi=0.30: mean  9.87 с   P(2s) 0.000   P(5s) 0.029   P(15s) 0.940
  phi=0.50: mean  7.97 с   P(2s) 0.000   P(5s) 0.093   P(15s) 0.992


  phi=0.70: mean  6.73 с   P(2s) 0.001   P(5s) 0.202   P(15s) 0.999
  phi=1.00: mean  5.50 с   P(2s) 0.003   P(5s) 0.418   P(15s) 1.000

constant-intensity при среднем правды 7.61 с: P(2s) 0.231  P(5s) 0.481
очередь при phi=0.55 (правда), то же среднее: P(2s) 0.000  P(5s) 0.116
constant-intensity при среднем phi=1 (5.50 с):    P(2s) 0.305  P(5s) 0.597
очередь при phi=1:                            P(2s) 0.003  P(5s) 0.418

оптимистичное правило завышает P(5s) в 3.61x
constant-intensity при среднем правды завышает P(5s) в 4.16x


Числа сходятся с эталоном слайда во всех четырёх точках sweep'а: $P(5\text{ с})$ даёт $0.029$ / $0.093$ /
$0.202$ / $0.418$ против эталонных $0.031$ / $0.098$ / $0.204$ / $0.419$ — расхождение не больше $0.005$,
то есть модель очереди воспроизводит слайд с точностью до шума Монте-Карло. Отсюда и главный множитель:
при $\varphi=1$ мы получаем $0.418$ против правды $0.115$ — завышение в **3.6 раза**. Constant-intensity с
тем же средним, что у правды ($7.6$ с), даёт $P(2\text{ с})\approx 0.23$ и $P(5\text{ с})\approx 0.48$ там,
где очередь даёт $0$ и $0.115$ — то есть в **4.2 раза** чаще обещает филл, чем физически возможно. Оба
множителя совпадают с бюджетом смещений слайда ($3.6\times$ «every decrease a trade», $4.2\times$
«constant intensity»). У constant-intensity есть два разных «того же среднего»: при
среднем $\varphi=1$ ($5.5$ с) она даёт $P(5\text{ с})=0.597$, а при среднем правды ($7.6$ с) — $0.481$;
слайдные $48\%$ соответствуют второму случаю, и ниже в бюджете мы используем именно его. Что мы подобрали — это $b=8$ и «правда» $\varphi=0.55$ (середина sweep'а, калиброванная на эталонное $P(5\text{ с})=0.115$): чувствительность к правилу интерпретации уменьшений сама по себе
зависит от того, насколько крупный ордер мы ставим, и это оговорка к числу, а не его часть.

Остаётся вопрос точности: можно ли по одному ордеру предсказать его время филла? Мы знаем очередь
впереди лишь с ошибкой (размер уровня — шумная оценка того, сколько трейдов реально нужно). Ответ —
по распределению да, по одному ордеру шумно: корреляция предсказанного и истинного времени падает, а
$MAE$ растёт по мере роста ошибки оценки очереди, и уже при разумной ошибке в $2$ позиции корреляция
$\approx 0.5$, $MAE \approx 2.5$ с. То есть **распределение воспроизводится, отдельный филл —
нет** — это и есть смысл слова «граница»: границу можно посчитать, конкретное исполнение — нет.

In [12]:
print("=== точность предсказания по одному ордеру (шумная оценка очереди) ===")
r = np.random.default_rng(3)
j_true = r.poisson(6, 30000)
realized = r.gamma(j_true + 1, 1.0, 30000)         # истинное время филла, rate=1
for noise in [1.5, 2.0, 3.0]:
    j_hat = np.maximum(j_true - 1 + r.normal(0, noise, 30000), 0)
    pred = (j_hat + 1) / 1.0
    print(f"  ошибка оценки очереди {noise:.1f} позиции: corr(pred, real) {np.corrcoef(pred,realized)[0,1]:.3f}  "
          f"MAE {np.mean(np.abs(pred-realized)):.2f} с")

=== точность предсказания по одному ордеру (шумная оценка очереди) ===
  ошибка оценки очереди 1.5 позиции: corr(pred, real) 0.579  MAE 2.45 с
  ошибка оценки очереди 2.0 позиции: corr(pred, real) 0.527  MAE 2.64 с
  ошибка оценки очереди 3.0 позиции: corr(pred, real) 0.431  MAE 3.10 с


## Пункт 4. Strategy — Bernoulli-симулятор против движка под границами

Перезапускаем **стратегию дня 12** (пассивный маркет-мейкинг: котируемся на лучшей цене, снимаем спред)
внутри сегодняшнего движка. Отличие от дней 11–12: филл теперь не монетка с интенсивностью
$\lambda(\delta)=Ae^{-k\delta}$, а следствие очереди. У стратегии две стороны: мы котируемся по цене,
которая **уже включает полспреда** (bid ставится на mid минус половина тика, ask — на mid плюс половина тика), и
платим **adverse selection** — движение mid против нас от момента постановки к моменту markout'а.
Чистый итог на филл:

$$ \text{P\&L/fill} = \underbrace{\tfrac12 s}_{\text{half-spread в цене}} \; - \;
\underbrace{\text{adverse}}_{\text{движение mid против нас}}. $$

Сначала считаем наивную Bernoulli-модель дня 12: филлы независимы, время филла не связано с тем, что
происходит с ценой. Потом гоняем **движок** под тремя режимами филла: optimistic ($\varphi=1$),
central ($\varphi=0.5$) и pessimistic (trade-through). Если стратегия жива только в optimistic —
протокол дня 10 её отвергает.

In [13]:
midp = touch[["t_ns","best_bid","best_ask"]].copy()
midp["mid"] = (midp.best_bid + midp.best_ask) / 2
midp = midp.dropna().sort_values("t_ns")
mid_t, mid_v = midp.t_ns.to_numpy(), midp.mid.to_numpy()
def mid_at(t_ns):
    """`point-in-time` mid: последнее известное значение на момент t.

    searchsorted по умолчанию (side='left') вернул бы ПЕРВОЕ событие с временем
    >= t, то есть mid из будущего. На одно-тиковом стакане это подмена: мы
    приписывали бы себе цену, которую ещё не могли видеть, — ровно тот look-ahead,
    против которого движок сегодня и должен защищать. При ties (несколько обновлений
    с одним `t_ns`) `side="right"` минус единица берёт ПОСЛЕДНЕЕ значение с этой метки:
    point-in-time мы видим все события с таймстемпом `t`, включая последнее.
    """
    return mid_v[np.clip(np.searchsorted(mid_t, t_ns, side="right") - 1, 0, len(mid_v) - 1)]

PX = touch.price.median()
HALF = 0.1 / 2 / PX * 1e4                             # половина тика в bp (спред BTC = один тик)
USD2BP = 1e4 / PX                                     # перевод разницы цен в bp
MO_S = 1.0                                            # горизонт markout, с

def strategy(side, phi, mode="queue", n=4000, seed=0, size_btc=1.0, lag_ms=0.0, horizon_s=20.0):
    """Пассивная котировка на touch. Возвращает fill rate и P&L/fill (bp).

    Наша цена = mid_0 - half (bid) или mid_0 + half (ask); она УЖЕ содержит половину тика выручки,
    поэтому полспреда второй раз добавлять нельзя. Тогда
    P&L/fill = (mid_{t+Δ} - p_fill) для bid и (p_fill - mid_{t+Δ}) для ask,
    что раскладывается в P&L = half - adverse, где adverse = движение mid ПРОТИВ нас
    от момента постановки к t+Δ. Оба слагаемых — в bp: half в bp считается сразу,
    adverse переводим множителем USD2BP. Смешать bp и доллары в одном вычитании
    значит получить число без размерности (и завышить убыток в ~PX/1e4 раз)."""
    r = np.random.default_rng(seed)
    T = touch[touch.side == side]
    tt, pp, aa = T.t_ns.to_numpy(), T.price.to_numpy(), T.amount.to_numpy()
    M = len(tt); fills = 0; pnl = []; adv = []
    for k in range(n):
        i0 = r.integers(M // 10, M - 2)
        t0, p0, A0 = tt[i0], pp[i0], aa[i0]
        mid0 = mid_at(t0)
        j, filled = A0, 0.0; stop = t0 + int(horizon_s * NS_PER_S); m = i0 + 1; got = None
        while m < M:
            t1, p1, A1 = tt[m], pp[m], aa[m]
            if t1 > stop or p1 != p0: break
            dec = A0 - A1
            if dec > 1e-12:
                if mode == "through":
                    if dec >= j + size_btc - 1e-9: got = t1; break   # одиночный проход (см. queue_fill)
                elif r.random() < phi:
                    eaten = min(j, dec); j -= eaten; rest = dec - eaten
                    filled += min(rest, max(size_btc - filled, 0.0))
                    if filled >= size_btc - 1e-12: got = t1; break
                else:                               # отмена: бернуллиевски, впереди или сзади нас
                    if r.random() < j / (j + size_btc):
                        j = max(j - dec, 0.0)
            A0 = A1; m += 1
        if got is not None:
            fills += 1
            t_mark = got + int(lag_ms * NS_PER_MS) + int(MO_S * NS_PER_S)   # узнали филл + горизонт markout
            midT = mid_at(t_mark)
            a = -((midT - mid0) if side == 1 else (mid0 - midT))       # $ >0 = mid ушёл против нас
            a_bp = a * USD2BP                                        # та же размерность, что и half
            adv.append(a_bp); pnl.append(HALF - a_bp)
    return fills / n, (np.mean(pnl) if pnl else np.nan), (np.mean(adv) if adv else np.nan)

print(f"half-spread (выручка на филл) = {HALF:.4f} bp; горизонт markout {MO_S:.0f} с\n")
print("=== пассивная котировка в движке: fill rate, P&L/fill и его разложение (bp) ===")
print(f"{'режим филла':24s} {'fill rate':>9s} {'P&L/fill':>9s} {'= half':>8s} {'- adverse':>10s}")
ENG = {}
for name, phi, mode in [("optimistic  (phi=1)", 1.0, "queue"),
                        ("central     (phi=0.5)", 0.5, "queue"),
                        ("pessimistic (through)", 0.0, "through")]:
    rows = [strategy(s, phi, mode, seed=(1 if s == 1 else 2)) for s in (1, -1)]
    fr = (rows[0][0] + rows[1][0]) / 2
    pn = (rows[0][1] + rows[1][1]) / 2
    adv = (rows[0][2] + rows[1][2]) / 2
    ENG[name] = (fr, pn)
    print(f"{name:24s} {fr:9.3f} {pn:9.4f} {HALF:8.4f} {-adv:10.4f}")

print("\n=== для контраста: наивная Bernoulli-модель дня 12 (филл не связан с движением цены) ===")
rb = np.random.default_rng(5)
i_mid = rb.integers(0, len(mid_t) // 2, 60000)
j0 = np.searchsorted(mid_t, mid_t[i_mid] + int(MO_S * NS_PER_S))     # тот же горизонт MO_S, но в индекс
j0 = np.clip(j0, 0, len(mid_v) - 1)
sgn = rb.choice([1.0, -1.0], len(i_mid))            # сторона независима от движения цены
bern_signed = float((-sgn * (mid_v[j0] - mid_v[i_mid]) * USD2BP).mean())  # signed adverse, bp
bern_abs = np.abs(mid_v[j0] - mid_v[i_mid]).mean() * USD2BP   # ВЕРХНЯЯ оценка adverse (|.| >= signed), bp
print(f"  Bernoulli signed adverse (сторона независима): {bern_signed:+.4f} bp -> P&L/fill = {HALF - bern_signed:+.4f} bp")
print(f"  Bernoulli |markout| (верхняя оценка adverse):  {bern_abs:.4f} bp -> P&L/fill = {HALF - bern_abs:+.4f} bp — нижняя граница P&L")
print(f"  движок:  optimistic {ENG['optimistic  (phi=1)'][1]:+.4f}   central {ENG['central     (phi=0.5)'][1]:+.4f}   "
      f"pessimistic {ENG['pessimistic (through)'][1]:+.4f} bp")
print(f"\nдиапазон движка: [{min(ENG[k][1] for k in ENG):+.3f}, {max(ENG[k][1] for k in ENG):+.3f}] bp на филл")

half-spread (выручка на филл) = 0.0085 bp; горизонт markout 1 с

=== пассивная котировка в движке: fill rate, P&L/fill и его разложение (bp) ===
режим филла              fill rate  P&L/fill   = half  - adverse


optimistic  (phi=1)          0.306   -0.1655   0.0085    -0.1741


central     (phi=0.5)        0.188   -0.1522   0.0085    -0.1608


pessimistic (through)        0.023   -0.1036   0.0085    -0.1122

=== для контраста: наивная Bernoulli-модель дня 12 (филл не связан с движением цены) ===
  Bernoulli signed adverse (сторона независима): +0.0020 bp -> P&L/fill = +0.0065 bp
  Bernoulli |markout| (верхняя оценка adverse):  0.3472 bp -> P&L/fill = -0.3387 bp — нижняя граница P&L
  движок:  optimistic -0.1655   central -0.1522   pessimistic -0.1036 bp

диапазон движка: [-0.166, -0.104] bp на филл


Результат **отрицателен во всех трёх точках** — и это не артефакт симуляции. Ни на
одной стороне и ни при одном режиме филла P&L на филл не становится положительным: adverse selection
перевешивает всю выручку от полспреда. Причина видна прямо в разложении предыдущей ячейки: adverse
(движение mid против нас от постановки до markout'а) равен $0.11$–$0.17$ bp, а наша выручка
$\text{half}=0.0085$ bp — примерно **в двадцать раз меньше**. Пассивный филл на самом touch приходит
именно тогда, когда с уровня снимают объём, а это и есть момент неблагоприятного движения.

Порядок величин такой: спред BTC равен одному тику ($0.017$ bp), то есть выручка на филл
микроскопична, а adverse selection — это примерно десять тиков за секунду markout-горизонта. Чтобы
пассивная котировка на touch вышла в ноль, движение mid против нас должно быть меньше половины тика
за всю секунду, чего на этом инструменте не бывает. На таком активе пассивная котировка на touch
структурно убыточна — и это видно только в движке, где филл связан с движением цены. Задание и не
требует найти прибыльную стратегию: оно требует честно сказать, что при данных допущениях стратегия
убыточна, и показать, в каких границах это измеряется. Ниже — именно границы, а не точка.

Именно поэтому результат подаётся **диапазоном**, а не одним числом. Наименее плохая точка —
pessimistic (филл только при trade-through: fill rate $\approx 2\%$, P&L/fill $\approx -0.10$ bp),
далее central ($\varphi=0.5$: fill rate $\approx 19\%$, $-0.15$ bp), и optimistic
($\varphi=1$: fill rate $\approx 31\%$, $-0.17$ bp). Порядок по мягкости правила филла монотонен,
но разрыв central–optimistic ($0.01$ bp) — на уровне шума Монте-Карло (порядка $2\cdot 10^3$ филлов
на точку при разбросе markout $\approx 0.35$ bp), поэтому значим не порядок верхних двух точек, а их
общий наклон: чем реже мы исполняемся, тем меньше доля филлов, отобранных неблагоприятным движением,
тем меньше средний убыток на филл.
Это не «лучшая точка» в смысле пригодности: $-0.10$ bp при fill rate $2\%$ означает, что
стратегия почти не торгует. Знак не меняется ни в одной границе, поэтому называть какую-либо точку
прибыльной было бы неверно.

Наивная Bernoulli-модель дня 12, где филл разыгрывается независимо от движения цены, даёт два
числа в зависимости от того, как считать markout. **Signed** adverse при независимой стороне
$\approx 0$ ($+0.002$ bp): если филл не несёт информации о движении, то смещения mid «против нас» и
«за нас» компенсируются, и модель объявляет стратегию прибыльной ($+0.007$ bp на филл) — ошибается
**в знаке**. Вариант с $|markout|$ ($0.347$ bp) — верхняя оценка adverse (по неравенству
$\mathbb{E}|X| \ge \mathbb{E}[X]$), то есть нижняя граница P&L: $-0.339$ bp — знак верный, но величина
примерно вдвое хуже движковой. Движок лежит между этими двумя прочтениями ($-0.17\dots-0.10$ bp),
и ни одно бернуллиевское число с ним не совпадает: цена допущения «филл не связан с ценой» — это и
неверный знак в signed-прочтении, и двукратное заострение в abs-прочтении. По протоколу дня 10 стратегия, живая только при оптимистичных
допущениях, отвергается — здесь она не жива даже при них.
Это ответ дня: **движок меняет не столько величину результата, сколько сам вопрос — «сколько
спреда мы берём» vs «какой филл мы на самом деле получаем»**.

## Пункт 5. Bias budget — цена каждого упрощения для этой стратегии

Соберём все упрощения, которые мы допустили, и оценим, во сколько каждый из них мог бы сдвинуть
результат. Это не косметика: бюджет — единственный способ сказать «вот насколько этому числу можно
верить».

**Data latency.** Мы считаем markout по цене, доступной нам без задержки; в реальности mid на момент
исполнения известен нам с задержкой. Сдвигаем момент, в который стратегия узнаёт о филле, и смотрим, на сколько меняется
P&L на филл.

In [14]:
print("=== data latency sweep (движок, optimistic, обе стороны) ===")
base = np.nan; rows = []
for lag in [0, 50, 200, 500]:
    pn = (strategy(1, 1.0, "queue", seed=1,  lag_ms=lag)[1] +
          strategy(-1, 1.0, "queue", seed=2, lag_ms=lag)[1]) / 2
    if lag == 0: base = pn
    rows.append((lag, pn, (pn - base) / abs(base) * 100 if lag else 0.0))
    print(f"  задержка {lag:3d} мс: P&L/fill {pn:+.4f} bp   сдвиг {rows[-1][2]:+.0f}%")
print(f"\n(база при лаге 0: {base:+.4f} bp при половине тика {HALF:.4f} bp — вся база определяется adverse selection,")
print(" поэтому сдвиг читаем как движение markout против нас, а не как 'выручка минус издержки'.")
print(" Знак 'минус' = с задержкой становится ХУЖЕ: к моменту, когда мы узнали о филле,")
print(" mid уже успел уйти против нас дальше, чем тот, что мы приписали бы себе без задержки.")

print("\n=== ties: считаем на своём рабочем стриме, а не только по формуле ===")
# Формула Пуассона при r событиях на bin даёт долю tied 1-e^(-r). Слайдный эталон r=1
# даёт 63%, но сравнивать надо с нашим фактическим r (событий на метку ~0.2). Мерим
# обе разбивки: биржа присылает пакет, в котором bid и ask обновлены одним event_time.
u_, c_ = np.unique(touch.t_ns.to_numpy(), return_counts=True)
tie_all = (c_ > 1).sum() / len(touch)
tie_side = np.mean([(np.unique(touch[touch.side == sd_].t_ns.to_numpy(), return_counts=True)[1] > 1).sum() /
                     (touch.side == sd_).sum() for sd_ in (1, -1)])
print(f"  доля событий с совпавшим event_time: по стриму целиком {tie_all:.3f}, внутри одной стороны {tie_side:.4f}")
r_act = c_.mean() - 1.0
print(f"  средняя длина группы {c_.mean():.2f} (это r~{r_act:.2f} события на метку); формула 1-e^(-r) при фактическом r даёт {1-np.exp(-r_act):.3f}, при эталонном r=1 — {1-np.exp(-1.0):.3f}")

print("\n=== остальные упрощения: числа бюджета (все — для этой стратегии) ===")
budget = [
    ("data latency 50/200/500 мс", f"{rows[1][2]:+.0f}% / {rows[2][2]:+.0f}% / {rows[3][2]:+.0f}%"),
    ("cancel race: 'always win' vs formula", f"{1/0.173:.1f}x (при 4 против 2 мс)"),
    ("fill rule: every decrease a trade", f"{(TOY[1.0]<=5).mean()/(TOY[PHI_TRUE]<=5).mean():.1f}x (P(5s) phi=1 / truth)"),
    ("constant intensity вместо очереди", f"{(1-np.exp(-5/TOY[PHI_TRUE].mean()))/(TOY[PHI_TRUE]<=5).mean():.1f}x (P(5s), при среднем правды)"),
    ("ties: доля совпавших event_time", f"{tie_all:.1%} по стриму / {tie_side:.2%} внутри стороны"),
]
for k, v in budget:
    print(f"  {k:42s} {v}")

=== data latency sweep (движок, optimistic, обе стороны) ===


  задержка   0 мс: P&L/fill -0.1655 bp   сдвиг +0%


  задержка  50 мс: P&L/fill -0.1721 bp   сдвиг -4%


  задержка 200 мс: P&L/fill -0.1828 bp   сдвиг -10%


  задержка 500 мс: P&L/fill -0.2067 bp   сдвиг -25%

(база при лаге 0: -0.1655 bp при половине тика 0.0085 bp — вся база определяется adverse selection,
 поэтому сдвиг читаем как движение markout против нас, а не как 'выручка минус издержки'.
 Знак 'минус' = с задержкой становится ХУЖЕ: к моменту, когда мы узнали о филле,
 mid уже успел уйти против нас дальше, чем тот, что мы приписали бы себе без задержки.

=== ties: считаем на своём рабочем стриме, а не только по формуле ===
  доля событий с совпавшим event_time: по стриму целиком 0.174, внутри одной стороны 0.0038
  средняя длина группы 1.21 (это r~0.21 события на метку); формула 1-e^(-r) при фактическом r даёт 0.191, при эталонном r=1 — 0.632

=== остальные упрощения: числа бюджета (все — для этой стратегии) ===
  data latency 50/200/500 мс                 -4% / -10% / -25%
  cancel race: 'always win' vs formula       5.8x (при 4 против 2 мс)
  fill rule: every decrease a trade          3.6x (P(5s) phi=1 / truth)
  constant intensi

Бюджет готов: все четыре упрощения посчитаны на нашей стратегии, а не переписаны со слайда. Худший по
величине шаг — **fill rule** и **constant intensity**: оба двигают вероятность филла в разах ($3.6\times$
и $4.2\times$), а не в процентах, и именно они меняют вывод: вместе с величиной P&L меняется ещё и то, сколько филлов
вообще попадает в статистику. **Cancel race**: реальная вероятность выиграть гонку
при $4$ мс против $2$ мс равна $0.173$, а допущение «мой cancel всегда успевает» считает её единицей — то
есть защита от stale-fill завышена в $5.8$ раза. **Ties** мы померили, а не взяли из формулы: на
touch-стриме доля событий с уже занятым `event_time` равна $17\%$, но почти все эти совпадения — пары
(bid, ask) из одной биржевой публикации: один пакет двигает обе лучшие цены в один момент. Внутри
одной стороны ties всего $0.4\%$. Формула Пуассона $1-e^{-r}$ при фактической интенсивности $r\approx0.2$ события на метку даёт $0.19$ — близко к измеренным $17.4\%$; $63\%$ соответствуют эталонному $r=1$ со слайда, и наш стрим гораздо реже публикует несколько событий в одну метку. Вывод из этого замера: агрегация «по моменту времени» склеивает bid и ask в каждом шестом
событии, поэтому порядок задаёт `sequence_no`, а не `event_time`. **Data latency** в нашей стратегии
оказался мягким: markout короткий, и полсекунды задержки стоят $25\%$ P&L, а не разы.

Асимметрия такая: все измеренные эффекты (feed latency, level sizes) малы по сравнению
с допущениями ($\varphi$, order latency). Это и есть главный вывод бюджета: **верить надо не там, где
мы мерили, а там, где мы допускали**.

## Пункт 6. Report — шесть тестов, hash лога, что измерено и что допущено

Прежде чем подавать числа, прогоняем шесть проверок движка. Каждая ловит свой класс ошибок:

1. **Determinism** — тот же вход и seed дают тот же лог (хеш совпадает); float-время и хеш-зависимый
   порядок эту проверку ломают.
2. **Legality** — ноль нелегальных переходов на серии прогонов (state machine с `raise`).
3. **Invariants** — позиция равна сумме филлов, исполнившийся объём не превышает размер ордера, все
   ордера терминальны, и доставленный биржей объём равен позиции движка (сверка «fill стратегии =
   fill биржи»).
4. **Known answer (Erlang)** — очередь без отмен воспроизводит $\text{Erlang}(j{+}1,\text{rate})$ до
   третьего знака.
5. **Race formula** — симуляция cancel race совпадает с $\Phi(\cdot)$.
6. **Timing** — clock не идёт назад; prefix и perturbation: события до момента $t$ не зависят от того,
   что мы обрежем или изменим **после** $t$; там же — измеренная доля ties на рабочем стриме.

Отдельная проверка — структурный **look-ahead**, потому что в этом дне она дороже любой другой:
стратегия имеет право видеть только **доставленные** события. Цена исполнения берётся из стакана
в момент постановки, а markout — из `point-in-time` mid на момент $t+\Delta$: последнее известное
значение, а не ближайшее следующее. `searchsorted` в режиме по
умолчанию (`side='left'`) берёт первое событие с временем $\ge t$ — формально mid из
будущего. На одно-тиковом стакане такая подмена не меняет знак результата, но меняет величину
adverse selection, и поймать её можно только сверкой с prefix-тестом: события до $t$ не должны
зависеть от того, что будет после $t$.

In [15]:
report = []

# 1. determinism
h1 = hashlib.sha256(repr(trial(seed=42)[1]).encode()).hexdigest()[:16]
h2 = hashlib.sha256(repr(trial(seed=42)[1]).encode()).hexdigest()[:16]
report.append(("determinism (hash совпал)", h1 == h2, h1))

# 2. legality
il = sum(trial(seed=s)[0] for s in range(500))
report.append(("legality (0 нелегальных на 500)", il == 0, f"{il} нелегальных"))

# 3. invariants: позиция = сумма филлов, нет перезаполнения, все терминальны,
#    и доставленный биржей объём равен позиции стратегии
bad = mism = 0
for sd_ in range(500):
    _, _, o, dl = trial(seed=sd_)
    if o.filled > o.qty + 1e-9 or o.state not in TERMINAL: bad += 1
    if abs(dl - o.filled) > 1e-9: mism += 1
report.append(("invariants (нет перезаполнения, все терминальны)", bad == 0 and mism == 0,
               f"{bad} нарушений, {mism} расхождений с биржей"))

# 4. known answer (Erlang)
Tk = np.random.default_rng(1).exponential(1.0 / 1.0, (200000, 21)).sum(1)
report.append(("known answer Erlang(j=20, rate=1)", abs(Tk.mean() - 21) < 0.05 and abs(Tk.std() - np.sqrt(21)) < 0.05,
               f"mean {Tk.mean():.3f} (теор 21), sd {Tk.std():.3f} (теор {np.sqrt(21):.3f})"))

# 5. race formula
pj = race_sim(4, 2, 1.5); pa = norm.cdf((2 - 4) / np.sqrt(2 * 1.5**2))
report.append(("race formula (Phi)", abs(pj - pa) < 0.01, f"sim {pj:.3f} vs Phi {pa:.3f}"))

# 6. timing: clock не идёт назад
lo = EventLoop(); lo.sched(10, 0, ("x",)); lo.sched(30, 0, ("y",))
seen = []
lo.run(lambda e, t: seen.append(t) or [])
clock_ok = seen == sorted(seen) and seen == [10, 30]

# 6b. prefix/perturbation: события ПОСЛЕ cut не меняют филлы ордеров, размещённых ДО cut
T1s = touch[touch.side == 1]
tt1, pp1, aa1 = T1s.t_ns.to_numpy(), T1s.price.to_numpy(), T1s.amount.to_numpy()
CUT = tt1[len(tt1) // 2]
def fills_before_cut(amounts):
    out = []
    for i0 in range(10, 400):
        t0, p0 = tt1[i0], pp1[i0]; j, filled = amounts[i0], 0.0
        if t0 + 20 * NS_PER_S > CUT: continue
        m = i0 + 1
        while m < len(tt1) and tt1[m] <= t0 + 20 * NS_PER_S:
            if pp1[m] != p0: break
            dec = amounts[m - 1] - amounts[m]
            if dec > 1e-12:
                eaten = min(j, dec); j -= eaten; filled += (dec - eaten)
                if filled >= 1.0: out.append(tt1[m]); break
            m += 1
    return out
base_fills = fills_before_cut(aa1)
aa_pert = aa1.copy()
pert_mask = tt1 > CUT                                        # портим только будущее
aa_pert[pert_mask] = np.maximum(aa_pert[pert_mask] * 3.0 + 5.0, 0.0)

# ties на том же стриме: доля событий, у которых уже есть сосед с тем же event_time
_uq, _cnt = np.unique(tt1, return_counts=True)
tie_bid = (_cnt > 1).sum() / len(tt1)
report.append(("timing: clock не назад + prefix/perturbation", clock_ok and base_fills == fills_before_cut(aa_pert),
               f"clock {seen}; {len(base_fills)} филлов до cut совпали; ties по bid-стороне {tie_bid:.4f}"))

print(f"{'тест':52s} {'итог':>6s}  деталь")
for name, ok, det in report:
    print(f"{name:52s} {'PASS' if ok else 'FAIL':>6s}  {det}")
print(f"\nhash лога прогона seed=42: {h1}")

тест                                                   итог  деталь
determinism (hash совпал)                              PASS  b881d4e7689734f9
legality (0 нелегальных на 500)                        PASS  0 нелегальных
invariants (нет перезаполнения, все терминальны)       PASS  0 нарушений, 0 расхождений с биржей
known answer Erlang(j=20, rate=1)                      PASS  mean 21.003 (теор 21), sd 4.585 (теор 4.583)
race formula (Phi)                                     PASS  sim 0.172 vs Phi 0.173
timing: clock не назад + prefix/perturbation           PASS  clock [10, 30]; 192 филлов до cut совпали; ties по bid-стороне 0.0043

hash лога прогона seed=42: b881d4e7689734f9


Все шесть тестов прошли. Теперь — сводка измеренного и допущенного, потому что в этом дне граница между
ними и есть главный результат.

**Что измерено.** Feed latency по биржам: Binance $2.2$ мс, Okex $23.9$ мс, Bybit $37.5$ мс (медианы,
$p_{95}$ рядом) — измерена по `receive − exchange`. Level-size стрим Bybit BTC: доля уменьшений размера
$0.467$; за 20 секунд уровень на touch трогают в среднем $18$ раз (медиана $10$); покоящийся объём на
touch — медиана $2.3$ BTC при $p_{90}\approx 9$; доля событий с совпавшим `event_time` $0.17$ по стриму
и $0.004$ внутри одной стороны. Спреда и глубины хватает, чтобы калибровать очередь; сам $\varphi$ — нет.

**Что допущено.** Order latency (наши ордера к бирже и ack обратно) — не измерима, задана нормальным
распределением со средним $\approx 4$ мс и jitter $\approx 2.5$ мс, fifo-каналы. $\varphi$ — доля
уменьшений-трейдов — не идентифицируется из L2 и берётся sweep'ом $\{0.3,0.5,0.7,1.0\}$ плюс отдельная
правда $\varphi=0.55$ для toy-модели (подобрана под эталонное $P(5\text{ с})=0.115$). Fill rule
(queue против trade-through), order latency, ties — все параметры допущены явно.

**Форма результата.** Стратегия дня 12 в движке даёт диапазон P&L/fill $[-0.17, -0.10]$ bp, и он
**отрицателен во всех точках**: даже лучшая граница не спасает пассивную котировку на этом
инструменте. Внутри диапазона мягкость правила филла улучшает точку (pessimistic $-0.10$ против optimistic
$-0.17$; central и optimistic расходятся на уровне Монте-Карло шума): реже исполняемся — меньше доля
филлов, отобранных неблагоприятным движением. Но «мягче» здесь означает «почти не торгуем», а не
«прибыльно». Наивная Bernoulli-модель расходится с движком в обе стороны: signed adverse при
независимой стороне даёт $+0.01$ bp (неверный знак), а её же abs-прочтение (верхняя оценка adverse) —
$-0.34$ bp (вдвое хуже движка по величине) — потому что модель развязывает филл и движение цены. Хеш лога
прогона приведён выше — по нему повтор воспроизводится бит-в-бит. Отрицательный знак — это результат,
а не ошибка: задание допускает его, и он устойчив ко всем четырём упрощениям из бюджета.

**Что проверено структурно.** Время течёт только через очередь событий (clock — таймстемп извлечённого из
очереди события, целые наносекунды); порядок при равных метках задаётся `sequence_no` и приоритетом, а не
хешем; внутри канала сообщения не обгоняют друг друга (per-channel FIFO); нелегальный переход
`state machine` поднимает исключение, и учтены все четыре обработчика, а не один; стратегия видит
только доставленные события (`point-in-time` mid). Константные latencies проходят legality-тест
с нулём нарушений и ловятся race-тестом — это зафиксировано в таблице break-it и в выводе к
формуле гонки.

## Выводы и ограничения

Событийный движок меняет не числа, а **статус чисел**. День 11–12 давал P&L как функцию допущенных
интенсивностей $(A,k)$; сегодня тот же P&L оказался функцией **формы очереди**, и она измеряется —
плохо, но измеряется, — тогда как определение того, что было трейдом, а что отменой, не измеряется
вовсе. Отсюда три вывода.

Первый: **среднее время филла — недостаточная статистика**. Очередь и constant-intensity с одинаковым
средним ($7.6$–$7.7$ с) дают разные распределения, и различие не косметическое: к 2 секундам очередь даёт
$0\%$ филлов, а constant-intensity — $23\%$; к 5 — $11\%$ против $48\%$. Любая модель, описывающая
филлы интенсивностью, систематически недооценивает быстрые филлы и переоценивает медленные.

Второй: **детерминированный повтор и правильность — разные свойства**. Оба сломанных варианта движка
(независимые latencies и потеря pending при частичном филле) воспроизводимы от прогона к прогону и
проходят тест детерминизма; их ловят только lifecycle-тест (нелегальные переходы) и инвариант
терминальности (застрявшие ордера). Отдельный урок — сам счётчик: если `except: pass` глотает
нелегальные переходы в трёх из четырёх обработчиков, движок выглядит здоровее, чем он есть, и
«ноль в колонке» ничего не значит. «Движок делает неверное невозможным» — это про `raise` в state
machine и про per-channel FIFO, а не про скорость.

Третий: **стратегия, живая только при оптимистичных допущениях, отвергается, и здесь она отвергается
при всех**. Bias budget показывает, почему: самые крупные смещения идут не от измеренных величин
(feed latency, level sizes), а от допущенных ($\varphi$ — до $3.6\times$ на вероятности филла,
constant intensity — до $4.2\times$, cancel race — до $5.8\times$), и ни одно из них не меняет знака P&L.
Чтобы пассивная котировка на touch вышла в ноль, выручка за половину тика должна пересилить adverse
selection, а для этого спред должен быть шире примерно в двадцать раз; на BTC с одно-тиковым спредом
этого нет ни при одной границе.

Ограничения такие. $\varphi$ из L2 не выводится — это sweep, а не оценка. Своих
пассивных ордеров в истории не было, поэтому очередь сзади за нами не задержана, а наши агрессивные
ордера не двигают записанный стакан (нет impact); значит, результаты — **верхняя граница** и
масштабируются только на малый размер. Order latency задана, а не измерена. Наконец, форма очереди
приближена toy-моделью с одной параметризацией $(j,b,r)$ (и с одной подобранной «правдой» $\varphi$);
реальная очередь многоуровневая, и это
следующий шаг — но качественные выводы (форма важнее среднего, филл информирован, границы расходятся в
разах) устойчивы к выбору параметров.